In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
import glob
import os
from tqdm import tqdm
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages


In [45]:
# Define the path the full dataset file
file_path = 'Input/diabetic_data.csv'

# Read the CSV file into a DataFrame
full_dataset = pd.read_csv(file_path)

full_dataset


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [46]:
# Get summary of the dataset
full_dataset.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [47]:
# Get the unique values for each column and their counts

for col in list(full_dataset.columns):
    counts = full_dataset[col].value_counts()
    print(counts)
    print("==================================")

encounter_id
2278392      1
190792044    1
190790070    1
190789722    1
190786806    1
            ..
106665324    1
106657776    1
106644876    1
106644474    1
443867222    1
Name: count, Length: 101766, dtype: int64
patient_nbr
88785891     40
43140906     28
1660293      23
88227540     23
23199021     23
             ..
11005362      1
98252496      1
1019673       1
13396320      1
175429310     1
Name: count, Length: 71518, dtype: int64
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64
weight
?            98569
[75-100)      1336
[50-75)

Name: count, dtype: int64
glimepiride-pioglitazone
No        101765
Steady         1
Name: count, dtype: int64
metformin-rosiglitazone
No        101764
Steady         2
Name: count, dtype: int64
metformin-pioglitazone
No        101765
Steady         1
Name: count, dtype: int64
change
No    54755
Ch    47011
Name: count, dtype: int64
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


In [48]:
print("Missing values in each column\n\n")
print("==================================")

for col in list(full_dataset.columns):
    missing_count = full_dataset.loc[(full_dataset[col] == '?') | (full_dataset[col].isnull())].shape[0]
    if (missing_count > 0):
        print(col , ": " , missing_count)
        print('Missing percentage: ', round((missing_count/101766.0)*100, 2), "%")
        print("==================================")

Missing values in each column


race :  2273
Missing percentage:  2.23 %
weight :  98569
Missing percentage:  96.86 %
payer_code :  40256
Missing percentage:  39.56 %
medical_specialty :  49949
Missing percentage:  49.08 %
diag_1 :  21
Missing percentage:  0.02 %
diag_2 :  358
Missing percentage:  0.35 %
diag_3 :  1423
Missing percentage:  1.4 %
max_glu_serum :  96420
Missing percentage:  94.75 %
A1Cresult :  84748
Missing percentage:  83.28 %


In [49]:
# -------------------------
# High Missing Rate (~95%) 
# -------------------------

# Dropping weight & max_glu_serum. These have too little information to be reliable predictors. 
# Even imputation won't create meaningful signal from around 5% of observed data.

# ------------------------------
# Moderate Missing Rate (40-85%) 
# ------------------------------

# A1Cresult (83.28%)
# An A1C test measures the average amount of sugar in your blood over the past few months. Healthcare providers use 
# it to help diagnose prediabetes and Type 2 diabetes and to monitor how well your diabetes treatment plan is 
# working.

# medical_specialty (49.08%)
# This refers to the medical specialty of the admitting physician. This can be important as this can indirectly give
# information about pre-existing conditions.

# payer_code (39.56%)
# Integer identifier corresponding to 23 distinct values, for example, Blue Cross\Blue Shield, Medicare, and 
# self-pay. (However this was removed as a feature according to the PDF file)

# ------------------------------
# Low Missing Rate (< 3%) 
# ------------------------------

# race (2.23%): Impute with mode

# diag_1 (0.02%): Impute with mode

# diag_2 (0.35%): Impute with "None"

# diag_3 (1.4%): Impute with "None"


# # Handling missing values

# # 1. Drop high-missingness columns
# df_clean = df.drop(['weight', 'max_glu_serum'], axis=1)

# # 2. Handle medium-missingness categoricals
# df_clean['medical_specialty'].fillna('Unknown', inplace=True)
# df_clean['payer_code'].fillna('Missing', inplace=True)
# df_clean['A1Cresult'].fillna('Not_Tested', inplace=True)

# # 3. Handle low-missingness features
# df_clean['race'].fillna(df_clean['race'].mode()[0], inplace=True)
# df_clean['diag_1'].fillna(df_clean['diag_1'].mode()[0], inplace=True)
# df_clean['diag_2'].fillna('None', inplace=True)
# df_clean['diag_3'].fillna('None', inplace=True)


In [50]:
# Drop payer_code, weight, & max_glu_serum

df_clean = full_dataset.drop(['weight', 'max_glu_serum', 'payer_code'], axis=1)
df_clean 


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,?,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,?,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,?,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,?,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),1,3,7,3,?,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),1,4,5,5,?,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),1,1,7,1,?,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),2,3,7,10,Surgery-General,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [51]:
y = df_clean['readmitted'] # Target
X = df_clean.drop('readmitted', axis=1) # Features


In [52]:
from sklearn.model_selection import train_test_split

# We want to set aside 20% for final testing
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% goes to test
    random_state=42,     # Reproducibility
    stratify=y           # Keep same class proportions
)

print("Datapoints for training & validation: ", X_temp.shape[0])
print("Datapoints for testing (in lockbox): ", X_test.shape[0])

Datapoints for training & validation:  81412
Datapoints for testing (in lockbox):  20354


In [53]:
from collections import Counter
counts = Counter(list(y_temp.values))
total = len(list(y_temp.values))

print("Values in the 'readmitted' column in training & validation dataset")
print("------------------------------------------------------")

for value, count in counts.items():
    percent = count / total * 100
    print(f"{value}: {count} ({percent:.2f}%)")

Values in the 'readmitted' column in training & validation dataset
------------------------------------------------------
>30: 28436 (34.93%)
NO: 43891 (53.91%)
<30: 9085 (11.16%)


In [54]:
from collections import Counter
counts = Counter(list(y_test.values))
total = len(list(y_test.values))

print("Values in the 'readmitted' column in testing dataset")
print("------------------------------------------------------")

for value, count in counts.items():
    percent = count / total * 100
    print(f"{value}: {count} ({percent:.2f}%)")

Values in the 'readmitted' column in testing dataset
------------------------------------------------------
NO: 10973 (53.91%)
<30: 2272 (11.16%)
>30: 7109 (34.93%)


In [55]:
full_dataset.loc[full_dataset['diag_2'] == '?'][['encounter_id', 'diag_1', 'diag_2', 'diag_3']]

,encounter_id,diag_1,diag_2,diag_3
0,2278392,250.83,?,?
66,715086,250.11,?,?
216,2735964,250.03,?,?
263,2948334,250.8,?,?
431,3902532,250.13,?,?
...,...,...,...,...
99621,415526432,428,?,428
100559,427825172,599,?,41
100787,430828958,250.01,?,?
101192,436145102,781,?,250.02


In [56]:
full_dataset[['A1Cresult']]

,A1Cresult
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
101761,>8
101762,NaN
101763,NaN
101764,NaN


In [ ]:
# clustering for batch effects - PCA/UMAP/tSNE 
# Remove data points that are missing/none/?
# Send notebook to tobi
# do the preprocessing
